
# Persiapan Data Penyakit Kronis — Skrining Bantul 2025

Notebook ini melakukan **seluruh pembersihan dan penyiapan data**, lalu mengekspor dataset
siap pakai ke `Data Setiap Target/`. Masalah yang diperbaiki di sini sudah
didiagnosis lebih dulu di `Analisis Data Mentah.ipynb`.

Pembersihan di sini melayani **dua tujuan sekaligus**, bukan hanya modeling:

- **Analisis** — rata-rata IMT mentah adalah 69.5 karena beberapa error input yang ekstrem;
  tanpa pembersihan, tidak ada satu pun statistik deskriptif yang bisa dipercaya.
- **Modeling** — error yang sama juga merusak model.

Karena itu setiap keputusan pembersihan dicatat jumlah barisnya, supaya jejak auditnya
utuh dan bisa dipertanggungjawabkan di laporan.

## Kebijakan pembersihan

**Perbaiki dulu, buang yang tidak bisa diperbaiki.** Bila sebuah error punya penjelasan
mekanis yang jelas (tinggi & berat tertukar posisi saat input), baris itu **diperbaiki**.
Bila setelah perbaikan nilainya tetap mustahil, baris itu **dibuang** — bukan diimputasi,
bukan disimpan sebagai `NaN`.

## Alur

| Tahap | Isi |
|---|---|
| 1 | Standardisasi label target (semua kolom) |
| 2 | Perbaikan tinggi & berat badan yang tertukar |
| 3 | Rekalkulasi IMT |
| 4 | Buang antropometri yang tetap mustahil |
| 5 | Batas plausibilitas per kelompok usia |
| 6 | Klasifikasi kategori BMI (WHO + CDC) |
| 7 | Verifikasi hasil pembersihan |
| 8 | Pembentukan dataset per target |
| 9 | Ringkasan audit |
| 10 | Ekspor |

## Catatan tentang kolom `id`

Kolom `id` adalah **kunci sintetis yang sudah diacak** untuk melindungi data asli BPJS —
hanya ada 10.000 nilai unik untuk 350 ribu baris, dan satu `id` yang sama memuat kedua
jenis kelamin dengan puluhan usia berbeda. Jadi `id` **bukan** penanda individu.

Kolom ini tetap disertakan dalam hasil ekspor semata-mata untuk penelusuran baris, dan
**harus dibuang sebelum modeling**. Deduplikasi tingkat individu tidak mungkin dilakukan,
sehingga semua angka prevalensi nanti berlaku per catatan skrining, bukan per orang.

In [ ]:

# ===================== IMPORTS =====================
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modul bersama + konfigurasi wilayah. Akar proyek dicari dari folder kerja,
# jadi tidak ada satu pun jalur absolut di notebook ini: seluruh folder boleh
# dipindah atau diserahkan ke orang lain tanpa sel ini perlu diubah.
WILAYAH = "Bantul"

AKAR = Path.cwd()
while not (AKAR / "Bersama").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "Bersama"))

import konfigurasi as cfg      # jalur + cara baca berkas mentah tiap wilayah
import bmi_standards as bs      # standar BMI & batas plausibilitas per usia
import viz_style as vz          # gaya grafik seragam untuk semua notebook

vz.apply_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

FILEDATA = cfg.berkas_mentah(WILAYAH)
OUTPUT_DIR = cfg.folder_data(WILAYAH)

TARGETS = [
    "Berisiko Diabetes Melitus",
    "Berisiko Hipertensi",
    "Berisiko Kanker Paru",
    "Berisiko TBC",
]

# Jejak audit: tiap tahap mencatat jumlah baris yang tersisa.
audit = []
def catat(tahap, n, keterangan=""):
    audit.append({"Tahap": tahap, "Baris tersisa": n, "Keterangan": keterangan})
    print(f"{tahap:<45} {n:>9,}   {keterangan}")

In [ ]:

# ===================== LOAD DATA =====================
df = cfg.baca_mentah(WILAYAH)

n_awal = len(df)
catat("0. Data mentah", n_awal)

df_clean = df.copy()
display(df_clean.head())


## 1. Standardisasi Label Target

Typo diperbaiki di **semua** kolom target sekaligus. Pada versi sebelumnya perbaikan hanya
diterapkan ke `Berisiko Diabetes Melitus`, padahal deteksinya berjalan untuk semua kolom.
`.str.strip()` ditambahkan untuk membersihkan spasi tersembunyi.

In [ ]:

# ===================== STANDARDISASI LABEL =====================
LABEL_MAP = {"Tidah Berisiko": "Tidak Berisiko"}
VALID_LABELS = {"Berisiko", "Tidak Berisiko", "Tidak Dapat Pertanyaan"}

n_typo = int((df_clean[TARGETS] == "Tidah Berisiko").sum().sum())

for col in TARGETS:
    df_clean[col] = df_clean[col].replace(LABEL_MAP).str.strip()

# Verifikasi: pastikan tidak ada label lain yang tersisa.
for col in TARGETS:
    sisa = set(df_clean[col].dropna()) - VALID_LABELS
    print(f"{col:<30}: {'OK' if not sisa else sisa}")

# Standardisasi kolom kategorik lain.
df_clean["Jeniskelamin"] = df_clean["Jeniskelamin"].str.strip().str.upper()
df_clean["segmen"] = df_clean["segmen"].str.strip().str.upper()

print(f"\nTypo 'Tidah Berisiko' diperbaiki: {n_typo:,} sel")
print(f"Jenis kelamin di luar L/P       : {int((~df_clean['Jeniskelamin'].isin(['L','P'])).sum())}")


## 2. Perbaikan Tinggi & Berat Badan yang Tertukar

**Ini perbaikan penting dibanding versi sebelumnya.** Aturan lama mensyaratkan tinggi
*dan* berat sama-sama berada di luar rentang wajar sebelum ditukar. Padahal pada baris yang
benar-benar tertukar — misalnya `tinggi 54 cm / berat 159 kg` — nilai 54 masih sah sebagai
tinggi (bayi) dan 159 masih sah sebagai berat (obesitas). Keduanya lolos bila diperiksa
terpisah, sehingga aturan lama tidak pernah aktif dan baris-baris itu ikut terbuang.

Aturan baru memakai **IMT** sebagai penentu: sebuah baris ditukar hanya bila IMT-nya
mustahil sebagaimana tercatat, **tetapi** menjadi wajar setelah ditukar, **dan** kedua
nilainya masuk akal pada peran barunya. Tiga syarat ini membuat pertukaran hanya terjadi
pada kasus yang tidak ambigu.

In [ ]:

# ===================== PERBAIKAN TINGGI & BERAT =====================
# Tanda negatif diasumsikan salah input, bukan nilai bermakna.
df_clean["Tinggibadan"] = df_clean["Tinggibadan"].abs()
df_clean["Berat Badan"] = df_clean["Berat Badan"].abs()

h = df_clean["Tinggibadan"]
w = df_clean["Berat Badan"]

bmi_asis = w / (h / 100) ** 2      # IMT sebagaimana tercatat
bmi_swap = h / (w / 100) ** 2      # IMT bila kedua kolom ditukar

H_LO, H_HI = bs.GLOBAL_BOUNDS["tinggi"]
W_LO, W_HI = bs.GLOBAL_BOUNDS["berat"]

swap_candidate = (
    ~bmi_asis.between(bs.BMI_MIN, bs.BMI_MAX)      # mustahil apa adanya
    & bmi_swap.between(bs.BMI_MIN, bs.BMI_MAX)     # wajar setelah ditukar
    & w.between(H_LO, H_HI)                        # berat tercatat sah sebagai TINGGI
    & h.between(W_LO, W_HI)                        # tinggi tercatat sah sebagai BERAT
)

n_swap = int(swap_candidate.sum())
print(f"Baris terindikasi tertukar (diperbaiki): {n_swap:,}")

print("\nContoh sebelum diperbaiki:")
display(
    df_clean.loc[swap_candidate, ["id", "Usia", "Tinggibadan", "Berat Badan"]]
            .assign(IMT_tercatat=bmi_asis[swap_candidate].round(1),
                    IMT_setelah_tukar=bmi_swap[swap_candidate].round(1))
            .head(8)
)

df_clean.loc[swap_candidate, ["Tinggibadan", "Berat Badan"]] = (
    df_clean.loc[swap_candidate, ["Berat Badan", "Tinggibadan"]].values
)

catat("2. Setelah perbaikan tukar tinggi/berat", len(df_clean), f"{n_swap:,} baris diperbaiki")


## 3. Rekalkulasi IMT

IMT dihitung ulang dari tinggi & berat yang sudah bersih. Kolom `Index Masa Tubuh` asli
tidak dipakai karena ikut tercemar oleh error input yang sama.

In [ ]:

# ===================== REKALKULASI IMT =====================
df_clean["IMT_recalculated"] = (
    df_clean["Berat Badan"] / (df_clean["Tinggibadan"] / 100) ** 2
).round(2)

# Seberapa dekat IMT asli dengan hasil hitungan ulang?
# Dipisah antara baris yang TIDAK disentuh Tahap 2 dan baris yang tinggi/beratnya
# baru saja ditukar -- mencampur keduanya akan mengaburkan maknanya.
selisih = (df_clean["Index Masa Tubuh"] - df_clean["IMT_recalculated"]).abs()

for label, mask in [("Baris tidak diubah di Tahap 2", ~swap_candidate),
                    ("Baris yang tinggi/beratnya ditukar", swap_candidate)]:
    s = selisih[mask]
    print(f"{label}")
    print(f"   n                        : {int(mask.sum()):,}")
    print(f"   selisih rata-rata        : {s.mean():.3f}")
    print(f"   baris dengan selisih > 1 : {int((s > 1.0).sum()):,}")
    print()


Hasilnya terbelah tajam, dan justru itu yang informatif:

- Pada baris yang **tidak disentuh** Tahap 2, selisihnya nyaris nol. Artinya sistem sumber
  **menurunkan** IMT dari tinggi & berat yang sama, bukan mengukurnya secara terpisah.
- Pada baris yang **tinggi/beratnya baru ditukar**, selisihnya besar — memang seharusnya
  begitu, karena IMT asli dihitung dari nilai yang salah, sedangkan IMT baru dihitung dari
  nilai yang sudah dibetulkan.

Konsekuensinya: kolom `Index Masa Tubuh` asli **tidak bisa dipakai untuk memvalidasi**
tinggi & berat, karena ia ikut salah setiap kali tinggi/berat salah. Ini juga alasan
mengapa deteksi pertukaran di Tahap 2 harus bersandar pada rentang plausibilitas, bukan
pada kecocokan dengan IMT asli.


## 4. Buang Antropometri yang Tetap Mustahil

Baris yang setelah upaya perbaikan masih berada di luar rentang wajar dibuang. Sesuai
kebijakan, ini pembuangan langsung — tidak diimputasi dan tidak disimpan sebagai `NaN`.

In [ ]:

# ===================== BUANG YANG TETAP MUSTAHIL =====================
h, w = df_clean["Tinggibadan"], df_clean["Berat Badan"]
imt = df_clean["IMT_recalculated"]

valid_global = (
    h.between(H_LO, H_HI)
    & w.between(W_LO, W_HI)
    & imt.between(bs.BMI_MIN, bs.BMI_MAX)
)

rincian = pd.DataFrame([
    ["Tinggi di luar %g-%g cm" % (H_LO, H_HI), int((~h.between(H_LO, H_HI)).sum())],
    ["Berat di luar %g-%g kg" % (W_LO, W_HI), int((~w.between(W_LO, W_HI)).sum())],
    ["IMT di luar %g-%g" % (bs.BMI_MIN, bs.BMI_MAX), int((~imt.between(bs.BMI_MIN, bs.BMI_MAX)).sum())],
], columns=["Kriteria", "Baris melanggar"])
display(rincian)
print("(kriteria bisa tumpang tindih; total unik di bawah ini)")

n_drop_global = int((~valid_global).sum())
df_clean = df_clean[valid_global].copy()

catat("4. Setelah buang antropometri mustahil", len(df_clean), f"{n_drop_global:,} baris dibuang")


## 5. Batas Plausibilitas per Kelompok Usia

Batas global berlaku untuk seluruh populasi dari bayi sampai lansia, sehingga terpaksa
sangat longgar — akibatnya anak usia 0 tahun setinggi 197 cm tetap lolos, karena 197 cm
adalah tinggi yang sah bagi orang dewasa.

Batas tambahan per kelompok usia (`bs.CHILD_BOUNDS`) menangkap sisa kasus ini. Baris yang
gagal dibuang, sesuai kebijakan.

In [ ]:

# ===================== BATAS PER KELOMPOK USIA =====================
valid_usia = bs.apply_child_bounds(df_clean)

rincian_usia = []
for (lo, hi), b in bs.CHILD_BOUNDS.items():
    in_band = df_clean["Usia"].between(lo, hi)
    bad = in_band & ~valid_usia
    rincian_usia.append({
        "Kelompok usia": f"{lo}-{hi}",
        "Batas tinggi (cm)": f"{b['tinggi'][0]:.0f}-{b['tinggi'][1]:.0f}",
        "Batas berat (kg)": f"{b['berat'][0]:.0f}-{b['berat'][1]:.0f}",
        "N": int(in_band.sum()),
        "Dibuang": int(bad.sum()),
        "%": round(bad.sum() / max(in_band.sum(), 1) * 100, 2),
    })
display(pd.DataFrame(rincian_usia))

print("Contoh baris yang dibuang:")
display(
    df_clean[~valid_usia].nlargest(6, "Tinggibadan")[
        ["id", "Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]
    ]
)

n_drop_usia = int((~valid_usia).sum())
df_clean = df_clean[valid_usia].copy()

catat("5. Setelah batas per kelompok usia", len(df_clean), f"{n_drop_usia:,} baris dibuang")

## 5b. Baris Tanpa Segmen Kepesertaan

`segmen` (kategori kepesertaan BPJS) adalah fitur wajib bagi keempat model dan
sekaligus variabel utama pada analisis TBC. Baris tanpa `segmen` tidak dapat
dimodelkan maupun masuk ke tabel prevalensi per segmen.

Baris seperti ini **dibuang, bukan diisi** -- tidak ada dasar untuk menebak
kepesertaan seseorang dari tinggi, berat, atau usianya, dan mengisinya dengan
kategori terbanyak akan mengarang data pada variabel yang justru sedang diteliti.

Tahap ini dijalankan untuk **kedua wilayah** meskipun Bantul tidak memiliki satu
pun baris kosong, supaya jejak auditnya sebanding dan supaya ekspor berikutnya
yang kebetulan punya banyak baris kosong terlihat di angka audit, bukan lolos
diam-diam sampai tahap pemodelan.

In [ ]:

# ===================== BUANG BARIS TANPA SEGMEN =====================
n_segmen_kosong = int(df_clean["segmen"].isna().sum())
print(f"Baris tanpa segmen: {n_segmen_kosong:,}")

if n_segmen_kosong:
    display(
        df_clean.loc[
            df_clean["segmen"].isna(),
            ["id", "Jeniskelamin", "Usia", "Tinggibadan", "Berat Badan"],
        ].head(10)
    )

df_clean = df_clean[df_clean["segmen"].notna()].copy()

catat("5b. Setelah buang segmen kosong", len(df_clean),
      f"{n_segmen_kosong:,} baris dibuang")


## 6. Klasifikasi Kategori BMI

Klasifikasi diambil dari modul bersama `bmi_standards.py` supaya definisinya tidak
tercecer di banyak notebook. Sumber tabel: <https://www.calculator.net/bmi-calculator.html>.

| Kelompok usia | Standar | Kategori |
|---|---|---|
| `> 20 th` | **WHO** (dewasa) | Severe / Moderate / Mild Thinness, Normal, Overweight, Obese Class I/II/III |
| `2 - 20 th` | **CDC** BMI-for-age percentile | `Anak_Underweight`, `Anak_Healthy weight`, `Anak_At risk of overweight`, `Anak_Overweight` |
| `< 2 th` | — | `Balita (0-2th)`, sengaja tidak diklasifikasi |

**Dua hal yang perlu dicatat:**

1. **Label anak diberi awalan `Anak_`.** Tanpa awalan ini, `Overweight` berarti dua hal
   berbeda dalam satu kolom yang sama — IMT 25-30 untuk dewasa, tetapi persentil >95 untuk
   anak. Versi sebelumnya mencampur keduanya, sehingga setiap angka *feature importance*
   pada kolom kategori BMI menjadi tidak dapat dipercaya.

2. **Usia di bawah 2 tahun tidak diklasifikasi.** BMI-for-age tidak berlaku secara medis
   pada kelompok ini; WHO merekomendasikan *Weight-for-Length*, dan tabel rujukannya tidak
   tersedia dengan sumber yang dapat disitasi. Kelompok ini **tetap dipertahankan** di
   dataset (mereka nyata dan relevan untuk TBC), hanya saja kategori BMI-nya dikosongkan
   secara eksplisit alih-alih dipaksakan ke kategori yang tidak berlaku.

Dua kolom dihasilkan: `Kategori_BMI_WHO8` (8 kategori penuh) dan `Kategori_BMI_WHO4`
(diringkas jadi Underweight/Normal/Overweight/Obese), supaya tahap modeling bisa memilih
tingkat granularitasnya.

In [ ]:

# ===================== KLASIFIKASI BMI =====================
df_clean["Kategori_BMI_WHO8"] = bs.classify_bmi_series(
    df_clean["IMT_recalculated"], df_clean["Usia"], df_clean["Jeniskelamin"]
)
df_clean["Kategori_BMI_WHO4"] = df_clean["Kategori_BMI_WHO8"].map(bs.collapse_who4)

print("Kategori_BMI_WHO8:")
print(df_clean["Kategori_BMI_WHO8"].value_counts(dropna=False).to_string())
print("\nKategori_BMI_WHO4:")
print(df_clean["Kategori_BMI_WHO4"].value_counts(dropna=False).to_string())

# Uji regresi: satu label tidak boleh muncul di dewasa DAN anak sekaligus.
usia_per_label = df_clean.groupby("Kategori_BMI_WHO8")["Usia"].agg(["min", "max"])
tabrakan = usia_per_label[
    (usia_per_label["min"] <= bs.CHILD_MAX_AGE) & (usia_per_label["max"] > bs.CHILD_MAX_AGE)
]
print(f"\nLabel yang dipakai lintas kelompok usia (harus kosong): {list(tabrakan.index)}")


## 7. Verifikasi Hasil Pembersihan

In [ ]:

# ===================== SEBELUM vs SESUDAH =====================
fig, axes = plt.subplots(2, 4, figsize=vz.FIG_2X4)

raw_cols = ["Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]
clean_cols = ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]

for i, col in enumerate(raw_cols):
    lo, hi = df[col].quantile([0.01, 0.99])
    sns.histplot(df[col].clip(lo, hi), bins=40, kde=True, ax=axes[0, i], color="lightcoral")
    axes[0, i].set_title(f"{col} — SEBELUM")

for i, col in enumerate(clean_cols):
    sns.histplot(df_clean[col], bins=40, kde=True, ax=axes[1, i], color="mediumseagreen")
    axes[1, i].set_title(f"{col} — SESUDAH")

plt.tight_layout()
plt.show()

banding = pd.DataFrame({
    "SEBELUM (mentah)": df[raw_cols].mean().round(2).values,
    "SESUDAH (bersih)": df_clean[clean_cols].mean().round(2).values,
}, index=["Usia", "Tinggi Badan", "Berat Badan", "IMT"])
display(banding)


## 8. Pembentukan Dataset per Target

Baris berlabel `Tidak Dapat Pertanyaan` dibuang **per target**. Ini bukan penanganan data
hilang: responden tersebut memang **tidak ditanyai** karena berada di luar sasaran usia
instrumen skrining. Akibatnya setiap dataset mencakup rentang usia yang berbeda, dan hal
itu ikut dicatat di ringkasan.

**Encoding tidak dilakukan di sini.** `segmen` dan kedua kolom kategori BMI diekspor
sebagai teks apa adanya; one-hot encoding dipindahkan ke dalam pipeline di notebook
modeling. Dengan begitu tidak perlu lagi trik `pd.Categorical` untuk menjaga konsistensi
kolom antar file, dan notebook analisis bisa membaca datanya dalam bentuk yang terbaca
manusia. Hanya kolom target yang dipetakan ke 0/1, karena scikit-learn memerlukannya.

In [ ]:

# ===================== PEMBENTUKAN DATASET PER TARGET =====================
FEATURES = [
    "id",                  # hanya untuk penelusuran; dibuang sebelum modeling
    "Jeniskelamin",
    "Usia",
    "segmen",
    "Tinggibadan",
    "Berat Badan",
    "IMT_recalculated",
    "Kategori_BMI_WHO8",
    "Kategori_BMI_WHO4",
]

DISEASE_TARGETS = {
    "diabetes": "Berisiko Diabetes Melitus",
    "hipertensi": "Berisiko Hipertensi",
    "kanker_paru": "Berisiko Kanker Paru",
    "tbc": "Berisiko TBC",
}

TARGET_MAPPING = {"Tidak Berisiko": 0, "Berisiko": 1}


def build_target_dataset(df_src, target_col):
    data = df_src[FEATURES + [target_col]].copy()
    data = data[data[target_col].isin(TARGET_MAPPING)]
    data[target_col] = data[target_col].map(TARGET_MAPPING).astype(int)
    return data


datasets = {}
ringkas_target = []

for name, target_col in DISEASE_TARGETS.items():
    d = build_target_dataset(df_clean, target_col)
    datasets[name] = d
    ringkas_target.append({
        "Dataset": name,
        "Baris": f"{len(d):,}",
        "Usia min": int(d["Usia"].min()),
        "Usia maks": int(d["Usia"].max()),
        "Tidak Berisiko": f"{int((d[target_col] == 0).sum()):,}",
        "Berisiko": f"{int((d[target_col] == 1).sum()):,}",
        "% Berisiko": round(d[target_col].mean() * 100, 2),
    })

display(pd.DataFrame(ringkas_target))

In [ ]:

# ===================== DISTRIBUSI KELAS PER TARGET (STACKED) =====================
komposisi = pd.DataFrame({
    name: datasets[name][col].value_counts().sort_index().rename(
        {0: "Tidak Berisiko", 1: "Berisiko"})
    for name, col in DISEASE_TARGETS.items()
}).T[["Tidak Berisiko", "Berisiko"]]

persen = (komposisi.div(komposisi.sum(axis=1), axis=0) * 100).round(2)

display(komposisi)
display(persen)

WARNA = [vz.PALETTE["RISK"]["Tidak Berisiko"], vz.PALETTE["RISK"]["Berisiko"]]
label_x = [f"{n}\n(usia {int(datasets[n]['Usia'].min())}-{int(datasets[n]['Usia'].max())})"
           for n in komposisi.index]

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

# Kiri: jumlah absolut bertumpuk -- memperlihatkan beda ukuran dataset.
komposisi.plot(kind="bar", stacked=True, ax=axes[0], color=WARNA)
axes[0].set_title("Jumlah Absolut per Kelas")
axes[0].set_ylabel("Jumlah catatan skrining")
axes[0].set_xticklabels(label_x, rotation=0)
axes[0].legend(title="Kelas")
vz.thousands(axes[0])

for i, n in enumerate(komposisi.index):
    axes[0].text(i, komposisi.loc[n].sum(), f"n={komposisi.loc[n].sum():,}",
                 ha="center", va="bottom", fontsize=10)
axes[0].margins(y=0.12)

# Kanan: proporsi 100% -- inilah yang menentukan strategi penanganan ketimpangan.
persen.plot(kind="bar", stacked=True, ax=axes[1], color=WARNA)
axes[1].set_title("Komposisi Relatif (100%)")
axes[1].set_ylabel("% dari dataset")
axes[1].set_xticklabels(label_x, rotation=0)
axes[1].set_ylim(0, 100)
axes[1].legend(title="Kelas", loc="center left", bbox_to_anchor=(1.01, 0.5))

for i, n in enumerate(persen.index):
    bawah = 0
    for lab in ["Tidak Berisiko", "Berisiko"]:
        v = persen.loc[n, lab]
        if v > 5:
            axes[1].text(i, bawah + v / 2, f"{v:.1f}%", ha="center", va="center",
                         fontsize=11, color="white", weight="bold")
        bawah += v

fig.suptitle("Distribusi Kelas Dataset Siap Pakai", y=1.00)
plt.tight_layout()
plt.show()


Ketimpangan kelas pada **Kanker Paru** dan terutama **TBC** sangat mencolok. Ini akan
ditangani di notebook modeling lewat pembobotan kelas dan penyetelan ambang keputusan,
bukan lewat resampling.

Perhatikan juga rentang usia pada judul tiap panel — keempat dataset **bukan** populasi
yang sama.


## 9. Ringkasan Audit

In [ ]:

# ===================== RINGKASAN AUDIT =====================
print("=" * 72)
print(f"RINGKASAN PEMBERSIHAN DATA — Skrining {WILAYAH} 2025")
print("=" * 72)

audit_df = pd.DataFrame(audit)
audit_df["Selisih"] = audit_df["Baris tersisa"].diff().fillna(0).astype(int)
display(audit_df)

print(f"\nBaris awal   : {n_awal:,}")
print(f"Baris akhir  : {len(df_clean):,}")
print(f"Total dibuang: {n_awal - len(df_clean):,} ({(n_awal - len(df_clean))/n_awal*100:.2f}%)")
print(f"Total diperbaiki (tanpa dibuang): {n_swap:,} tukar tinggi/berat + {n_typo:,} typo label")

print("\n[Kategori BMI]")
print(df_clean["Kategori_BMI_WHO8"].value_counts(dropna=False).to_string())

print("\n[Missing value pada kolom kunci]")
key_cols = ["Tinggibadan", "Berat Badan", "IMT_recalculated", "Kategori_BMI_WHO8"]
mv = df_clean[key_cols].isna().sum()
print("Tidak ada." if mv.sum() == 0 else mv.to_string())

In [ ]:

# ===================== WATERFALL PEMBERSIHAN =====================
fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
ax.bar(range(len(audit_df)), audit_df["Baris tersisa"], color="steelblue")
ax.set_xticks(range(len(audit_df)))
ax.set_xticklabels([t.split(". ", 1)[-1] for t in audit_df["Tahap"]],
                   rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Baris tersisa")
ax.set_title("Jumlah Baris di Setiap Tahap Pembersihan")
for i, v in enumerate(audit_df["Baris tersisa"]):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
ax.margins(y=0.12)
plt.tight_layout()
plt.show()


## 10. Ekspor Dataset

In [ ]:

# ===================== EKSPOR =====================
os.makedirs(OUTPUT_DIR, exist_ok=True)

MASTER_COLS = [
    "id", "Jeniskelamin", "Usia", "segmen",
    "Tinggibadan", "Berat Badan", "Index Masa Tubuh",
    "IMT_recalculated", "Kategori_BMI_WHO8", "Kategori_BMI_WHO4",
] + TARGETS

master = df_clean[MASTER_COLS].copy()
master_path = os.path.join(OUTPUT_DIR, "dataset_master.csv")
master.to_csv(master_path, index=False, encoding="utf-8-sig")
print(f"dataset_master.csv       -> {master.shape[0]:,} baris x {master.shape[1]} kolom")

for name, data in datasets.items():
    path = os.path.join(OUTPUT_DIR, f"dataset_{name}.csv")
    data.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"dataset_{name + '.csv':<17} -> {data.shape[0]:,} baris x {data.shape[1]} kolom")

# Jejak audit ikut diekspor supaya notebook analisis tidak perlu menghitung ulang
# tahapan pembersihan (satu sumber kebenaran).
audit_out = pd.DataFrame(audit)
audit_out.to_csv(os.path.join(OUTPUT_DIR, "audit_pembersihan.csv"),
                 index=False, encoding="utf-8-sig")
print(f"audit_pembersihan.csv    -> {len(audit_out)} tahap")

print(f"\nSemua file tersimpan di:\n{OUTPUT_DIR}")


## 11. Verifikasi Dataset Hasil Ekspor

Bagian ini membaca kembali file yang baru ditulis — bukan variabel di memori — lalu
menjalankan serangkaian pemeriksaan otomatis. Tujuannya supaya kesalahan ekspor ketahuan
di sini, bukan setelah muncul sebagai hasil model yang aneh.

In [ ]:

# ===================== BACA ULANG HASIL EKSPOR =====================
terbaca = {
    name: pd.read_csv(os.path.join(OUTPUT_DIR, f"dataset_{name}.csv"))
    for name in datasets
}
terbaca["master"] = pd.read_csv(os.path.join(OUTPUT_DIR, "dataset_master.csv"))

for name, d in terbaca.items():
    print("=" * 100)
    print(f"dataset_{name}.csv   —   {d.shape[0]:,} baris x {d.shape[1]} kolom")
    print("=" * 100)
    display(d.head())

In [ ]:

# ===================== PEMERIKSAAN OTOMATIS =====================
per_target = {k: v for k, v in terbaca.items() if k != "master"}
hasil_cek = []

def cek(nama, lulus, detail):
    # Status ditulis sebagai teks, bukan lewat DataFrame.style, supaya tetap
    # tampil benar tanpa jinja2 (tidak terpasang di environment ini).
    hasil_cek.append({"Pemeriksaan": nama,
                      "Status": "[ LULUS ]" if lulus else "[ GAGAL ]",
                      "Detail": detail})

# --- 1. Skema identik antar file (selain kolom target) ---
skema = {n: [c for c in d.columns if not c.startswith("Berisiko")]
         for n, d in per_target.items()}
acuan = skema["diabetes"]
sama = all(v == acuan for v in skema.values())
cek("Skema kolom identik antar dataset", sama,
    f"{len(acuan)} kolom fitur: {', '.join(acuan)}")

# --- 2. Tipe data konsisten ---
dtypes = {n: d[acuan].dtypes.astype(str).to_dict() for n, d in per_target.items()}
sama_dtype = all(v == dtypes["diabetes"] for v in dtypes.values())
cek("Tipe data konsisten antar dataset", sama_dtype,
    "semua kolom fitur bertipe sama di keempat file")

# --- 3. Level kategorik konsisten ---
# Penting: workaround pd.Categorical sudah dihapus, jadi konsistensi ini
# perlu diperiksa secara eksplisit.
lvl_segmen = {n: set(d["segmen"].unique()) for n, d in per_target.items()}
segmen_ok = all(v == lvl_segmen["diabetes"] for v in lvl_segmen.values())
cek("Level 'segmen' konsisten antar dataset", segmen_ok,
    f"{len(lvl_segmen['diabetes'])} level: {sorted(lvl_segmen['diabetes'])}")

# Kategori BMI TIDAK harus identik: hanya TBC yang memuat balita & anak kecil,
# karena target lain punya batas usia minimal. Yang diperiksa adalah subset.
master_lvl = set(terbaca["master"]["Kategori_BMI_WHO8"].unique())
subset_ok = all(set(d["Kategori_BMI_WHO8"].unique()) <= master_lvl
                for d in per_target.values())
cek("Kategori BMI tiap dataset subset dari master", subset_ok,
    f"master punya {len(master_lvl)} kategori")

# --- 4. Balita hanya di TBC ---
balita = {n: bs.LABEL_BALITA in set(d["Kategori_BMI_WHO8"]) for n, d in per_target.items()}
balita_ok = balita["tbc"] and not any(v for k, v in balita.items() if k != "tbc")
cek(f"'{bs.LABEL_BALITA}' hanya muncul di TBC", balita_ok,
    f"{ {k: v for k, v in balita.items()} }")

# --- 5. Rentang nilai numerik masuk akal ---
batas = {
    "Tinggibadan": bs.GLOBAL_BOUNDS["tinggi"],
    "Berat Badan": bs.GLOBAL_BOUNDS["berat"],
    "IMT_recalculated": (bs.BMI_MIN, bs.BMI_MAX),
    "Usia": (0, 120),
}
pelanggar = []
for n, d in per_target.items():
    for kol, (lo, hi) in batas.items():
        if not d[kol].between(lo, hi).all():
            pelanggar.append(f"{n}.{kol}")
cek("Semua nilai numerik dalam batas plausibel", not pelanggar,
    "tidak ada pelanggaran" if not pelanggar else f"melanggar: {pelanggar}")

# --- 6. Target hanya bernilai 0/1 ---
target_ok, rincian_t = True, []
for n, d in per_target.items():
    col = DISEASE_TARGETS[n]
    nilai = set(d[col].dropna().unique())
    if nilai != {0, 1} or d[col].isna().any():
        target_ok = False
    rincian_t.append(f"{n}={sorted(nilai)}")
cek("Target hanya bernilai {0, 1} tanpa NaN", target_ok, "; ".join(rincian_t))

# --- 7. Gerbang eligibilitas usia sesuai harapan ---
HARAPAN = {"diabetes": 15, "hipertensi": 15, "kanker_paru": 45, "tbc": 0}
gate_ok, rincian_g = True, []
for n, usia_min in HARAPAN.items():
    aktual = int(per_target[n]["Usia"].min())
    if aktual != usia_min:
        gate_ok = False
    rincian_g.append(f"{n}: {aktual} (harap {usia_min})")
cek("Batas usia sesuai gerbang eligibilitas", gate_ok, "; ".join(rincian_g))

# --- 8. Rekonsiliasi jumlah baris terhadap master ---
rekon_ok, rincian_r = True, []
for n, col in DISEASE_TARGETS.items():
    harap = int(terbaca["master"][col].isin(["Berisiko", "Tidak Berisiko"]).sum())
    aktual = len(per_target[n])
    if harap != aktual:
        rekon_ok = False
    rincian_r.append(f"{n}: {aktual:,} vs {harap:,}")
cek("Jumlah baris cocok dengan master", rekon_ok, "; ".join(rincian_r))

# --- 9. Tidak ada missing value ---
mv = {n: int(d.isna().sum().sum()) for n, d in terbaca.items()}
cek("Tidak ada missing value", sum(mv.values()) == 0, f"{mv}")

laporan = pd.DataFrame(hasil_cek)
print("=" * 110)
print("LAPORAN VERIFIKASI DATASET")
print("=" * 110)
for _, r in laporan.iterrows():
    print(f"{r['Status']}  {r['Pemeriksaan']}")
    print(f"           {r['Detail']}")
print("=" * 110)

n_gagal = int((laporan["Status"] == "[ GAGAL ]").sum())
print(f"{len(laporan) - n_gagal}/{len(laporan)} pemeriksaan lulus.")
if n_gagal:
    raise AssertionError(
        f"{n_gagal} pemeriksaan GAGAL — periksa laporan di atas sebelum lanjut ke modeling.")
print("Semua pemeriksaan lulus. Dataset siap dipakai untuk analisis dan modeling.")

In [ ]:

# ===================== RINGKASAN STRUKTUR TIAP DATASET =====================
ringkas_akhir = []
for name, d in terbaca.items():
    col = DISEASE_TARGETS.get(name)
    ringkas_akhir.append({
        "File": f"dataset_{name}.csv",
        "Baris": f"{len(d):,}",
        "Kolom": d.shape[1],
        "Usia min-maks": f"{int(d['Usia'].min())}-{int(d['Usia'].max())}",
        "N segmen": d["segmen"].nunique(),
        "N kategori BMI": d["Kategori_BMI_WHO8"].nunique(),
        "% Berisiko": round(d[col].mean() * 100, 2) if col else "-",
        "Ukuran (MB)": round(
            os.path.getsize(os.path.join(OUTPUT_DIR, f"dataset_{name}.csv")) / 1e6, 1),
    })

display(pd.DataFrame(ringkas_akhir))


### Selesai

Dataset siap dipakai oleh tiga notebook berikutnya:

- `Analisis Data Bersih.ipynb` — statistik deskriptif & profil populasi
- `Klasifikasi BMI.ipynb` — pendalaman klasifikasi WHO & CDC
- `Script penyakitan/Stacked Revisi.ipynb` — pemodelan

**Yang perlu diingat saat memakai dataset ini:**

1. Buang kolom `id` sebelum modeling — kolom itu teracak dan tidak membawa informasi.
2. `segmen` dan kolom kategori BMI masih berupa teks; encoding dilakukan di pipeline.
3. Setiap dataset punya rentang usia sendiri. Sebutkan rentang itu setiap kali melaporkan
   angka apa pun dari dataset yang bersangkutan.
4. Semua angka berlaku **per catatan skrining**, bukan per orang.